## import libraries

In [26]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [ ]:
from src.Data_preprocessing import preprocess
from src.Feature_Engineering import feature_Engineering
df=preprocess()
df= feature_Engineering(df)

In [28]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

import optuna

## Train Test Split

In [29]:
x=df.drop("charges",axis=1)
y=df["charges"]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

## pipeline [1]

**Encoding + linearRegression**

In [31]:
passthrough_features = [
    "age",
    "bmi",
    "children"
]

categorical_features = [
    "sex",
    "smoker",
    "region"
]

preprocessor_1 = ColumnTransformer(  
    transformers=[
        (
            "one_hot_encoding",
            OneHotEncoder(),
            categorical_features
        ),

        (
            "children_age_bmi",
            "passthrough",
            passthrough_features

        )
    ]
)

In [32]:
pipeline_1 = Pipeline([
    ("preprocessor", preprocessor_1),
    ("model", LinearRegression())
])

In [33]:
scores = cross_val_score(
    pipeline_1,
    X_train,
    y_train,
    cv=10,
    scoring="r2"  # NumPy Array.
)

print(f"R² Scores : {scores}")
print(f"Mean R²   : {scores.mean():.4f}")
print(f"Std R²    : {scores.std():.4f}")

R² Scores : [0.73541457 0.69696159 0.81850383 0.785235   0.78581745 0.62505557
 0.61028896 0.70647266 0.75466062 0.77900085]
Mean R²   : 0.7297
Std R²    : 0.0664


___

## pipeline [2]

**y_train_log**

In [34]:
y_train_log = np.log1p(y_train)

In [35]:
categorical_features = [
    "sex",
    "smoker",
    "region"
]

passthrough_features = ["children","age", "bmi"]

preprocessor_2 = ColumnTransformer(
    transformers=[
        (
            "categorical_encoding",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),

        (
            "children_age_bmi",
            "passthrough",
            passthrough_features
        )
    ]
)

In [36]:
pipeline_log = Pipeline([
    ("preprocessor", preprocessor_2),
    ("model", LinearRegression())
])

In [37]:
scores = cross_val_score(
    estimator=pipeline_log,
    X=X_train,
    y=y_train_log,
    cv=10,
    scoring="r2"
)

In [38]:
print("R² Scores")
print(scores)

print("-"*40)

print(f"Mean R² : {scores.mean():.4f}")
print(f"Std  R² : {scores.std():.4f}")

R² Scores
[0.71680049 0.70962273 0.83862218 0.77623575 0.77626384 0.72695108
 0.68152729 0.72704181 0.78490885 0.76716254]
----------------------------------------
Mean R² : 0.7505
Std  R² : 0.0438


**His performance improved.**

___

## pipeline [3]

**bmi_category only**

In [39]:
numeric_features = [
    "age"
]
categorical_features = [
    "sex",
    "smoker",
    "region",
    "bmi_category"
]
passthrough_features = [
    "children"
]


preprocessor_3 = ColumnTransformer(
    transformers=[
        (
            "one_hot_encoding",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),

        (
            "passthrough",
            "passthrough",
            numeric_features + passthrough_features
        )
    ]
)

In [40]:
pipeline_3 = Pipeline([
    ("preprocessor", preprocessor_3),
    ("model", LinearRegression())
])

In [41]:
y_train_log = np.log1p(y_train)
scores = cross_val_score(
    pipeline_3,
    X_train,
    y_train_log,
    cv=10,
    scoring="r2"
)

print(scores.mean())
print(scores.std())

0.7513822454094294
0.04293418058160457


**His performance improved just a little bit.**

____

## pipeline [4]

**bmi_category and bmi**

In [42]:
numeric_features = [
    "age",
    "bmi"
]

categorical_features = [
    "sex",
    "smoker",
    "region",
    "bmi_category"
]

passthrough_features = [
    "children"
]

preprocessor_4 = ColumnTransformer(
    transformers=[

        (
            "one_hot_encoding",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),

        (
            "passthrough",
            "passthrough",
            numeric_features + passthrough_features
        )
    ]
)



In [43]:
pipeline_4 = Pipeline([
    ("preprocessor", preprocessor_4),
    ("model", LinearRegression())
])

In [44]:
scores = cross_val_score(
    pipeline_4,
    X_train,
    y_train_log,
    cv=10,
    scoring="r2"
)

print(scores.mean())
print(scores.std())

0.7505690121992702
0.042686746144260275


**His performance dropped just a little bit.**

**Replacing the continuous BMI feature with clinically meaningful BMI categories improved the Linear Regression model. This suggests that the relationship between BMI and insurance charges is not purely linear, and categorical BMI groups better capture the effect that help linearRegression📝👌**

## 📌the best pipeline in LinearRegression is **three**  Encoding with bmi_category only and log to target

## RandomForestRegressor✅📝

In [45]:
pipeline_rf = Pipeline([
    ("preprocessor", preprocessor_4), #  4 the best  bmi and bmi_category
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

In [46]:
scores_rf = cross_val_score(
    pipeline_rf,
    X_train,
    y_train,
    cv=10,
    scoring="r2"
)

In [47]:
print(f"Mean R² : {scores_rf.mean():.4f}")
print(f"Std  R² : {scores_rf.std():.4f}")

Mean R² : 0.8233
Std  R² : 0.0455


## 📌Random Forest benefited from using both the continuous BMI feature and the engineered BMI category, indicating that tree-based models can exploit both the exact numerical values and the categorical grouping simultaneously

## XGBRegressor✅📝

In [48]:
pipeline_xgb = Pipeline([
    ("preprocessor", preprocessor_4),
    ("model", XGBRegressor(
        random_state=42
    ))
])

In [49]:
scores_xgb = cross_val_score(
    pipeline_xgb,
    X_train,
    y_train,
    cv=10,
    scoring="r2"
)

In [50]:
print(f"Mean R² : {scores_xgb.mean():.4f}")
print(f"Std  R² : {scores_xgb.std():.4f}")

Mean R² : 0.7864
Std  R² : 0.0475


**Random Forest is the best.**

## 📌Applying a logarithmic transformation to the target improved the performance of Linear Regression, but did not improve tree-based models such as Random Forest and XGBoost. This is expected because tree-based algorithms do not assume a linear relationship between features and the target and can naturally handle skewed target distributions.

## Select Preprocessor 4 and RandomForest Model✅✅